In [0]:
-- store rankings

CREATE OR REPLACE MATERIALIZED VIEW gold_store_rankings
AS
SELECT
    f.store_id,
    s.store_name,
    s.city,
    s.county,
    COUNT(*)                    AS transactions,
    SUM(f.sale_dollars)         AS total_revenue,
    SUM(f.bottles_sold)         AS total_bottles,
    SUM(f.gross_profit)         AS total_profit,
    AVG(f.sale_dollars)         AS avg_sale,
    RANK() OVER (ORDER BY SUM(f.sale_dollars) DESC) AS store_rank,
    NTILE(100) OVER (ORDER BY SUM(f.sale_dollars) DESC) AS revenue_percentile
FROM dev.silver.fct_sales_deduped f
JOIN dev.silver.dim_stores s
    ON  f.store_id = s.store_id
    AND f.sale_date >= s.__START_AT
    AND (f.sale_date < s.__END_AT OR s.__END_AT IS NULL)
GROUP BY f.store_id, s.store_name, s.city, s.county

In [0]:
-- pipeline health view

CREATE OR REPLACE MATERIALIZED VIEW pipeline_health AS
SELECT
    current_timestamp() AS snapshot_ts,
    (SELECT COUNT(*) FROM dev.silver.fct_sales)         AS fact_rows_raw,
    (SELECT COUNT(*) FROM dev.silver.fct_sales_deduped) AS fact_rows_deduped,
    (SELECT COUNT(*) FROM dev.silver.fct_sales)
        - (SELECT COUNT(*) FROM dev.silver.fct_sales_deduped) AS duplicate_count,
    (SELECT COUNT(DISTINCT store_id)  FROM dev.silver.dim_stores  WHERE __END_AT IS NULL) AS active_stores,
    (SELECT COUNT(DISTINCT item_id)   FROM dev.silver.dim_products WHERE __END_AT IS NULL) AS active_products,
    (SELECT COUNT(DISTINCT vendor_id) FROM dev.silver.dim_vendors  WHERE __END_AT IS NULL) AS active_vendors,
    (SELECT MIN(sale_date) FROM dev.silver.fct_sales_deduped) AS earliest_sale,
    (SELECT MAX(sale_date) FROM dev.silver.fct_sales_deduped) AS latest_sale